In [0]:
orders_df = spark.read.csv("/Volumes/workspace/olist_landing/raw_files/olist_orders_dataset.csv" ,
                           header=True,
                           inferSchema=True)
itens_df = spark.read.csv("/Volumes/workspace/olist_landing/raw_files/olist_order_items_dataset.csv" ,
                           header=True,
                           inferSchema=True)
order_payment_df = spark.read.csv("/Volumes/workspace/olist_landing/raw_files/olist_order_payments_dataset.csv" ,
                           header=True,
                           inferSchema=True)                           
                                                    

In [0]:
batch_6_orders = orders_df.limit(100)

display(
    batch_6_orders.select("order_id", "customer_id", "*")
)


In [0]:
import uuid 
from pyspark.sql import functions as F

id_mapping = [
    (row.order_id, str(uuid.uuid4())) for row in batch_6_orders.select("order_id").collect()
]

df_mapping = spark.createDataFrame(id_mapping, ["order_id", "new_order_id"]
)                             
display(df_mapping)

In [0]:
batch_6_orders_new = (batch_6_orders.alias("o")
                      .join(df_mapping.alias("m"), 
                            F.col("o.order_id") == F.col("m.order_id"), "inner")
                       .select(
                           F.col("m.new_order_id").alias("order_id"),
                           *[
                                F.col(f"o.{c}")
                                for c in batch_6_orders.columns
                                if c != "order_id"
                            ]                   
                         )
                    )
batch_6_item_new = (itens_df.alias("i")
                      .join(df_mapping.alias("m"), 
                            F.col("i.order_id") == F.col("m.order_id"), "inner")
                       .select(
                           F.col("m.new_order_id").alias("order_id"),
                           *[
                                F.col(f"i.{c}")
                                for c in itens_df.columns
                                if c != "order_id"
                            ]                   
                         )
                    )
batch_6_payment_new = (order_payment_df.alias("p")
                      .join(df_mapping.alias("m"), 
                            F.col("p.order_id") == F.col("m.order_id"), "inner")
                       .select(
                           F.col("m.new_order_id").alias("order_id"),
                           *[
                                F.col(f"p.{c}")
                                for c in order_payment_df.columns
                                if c != "order_id"
                            ]                   
                         )
                    )

In [0]:


batch_path = "/Volumes/workspace/olist_landing/batched_files"
#orders
( 
    batch_6_orders_new.write 
    .format("csv") 
    .mode("overwrite") 
    .option("header", "true") 
    .save(f"{batch_path}/batch_6/orders")
)
#itens
( 
    batch_6_item_new.write 
    .format("csv") 
    .mode("overwrite") 
    .option("header", "true") 
    .save(f"{batch_path}/batch_6/items")
)
#payment
( 
    batch_6_payment_new.write 
    .format("csv") 
    .mode("overwrite") 
    .option("header", "true") 
    .save(f"{batch_path}/batch_6/payments")
)